<a href="https://colab.research.google.com/github/NaziaAfreen015/CSC791-DLBA/blob/main/CSC791_DLBA_ResNet.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import tensorflow as tf
import torchvision.models as models
import torch.nn as nn
import torchvision.transforms as transforms
from PIL import Image
import requests
from io import BytesIO
import torch
import torch.optim as optim
import torch.nn.utils.prune as prune
from torch.utils.data import DataLoader
from torchvision import datasets

import copy
import math
from typing import List, Tuple
import time


import cv2
import matplotlib.pyplot as plt

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


## Preprocess Train

In [3]:
def preprocess_image(img_path):
  # Define the required transformations
  preprocess = transforms.Compose([
      transforms.Resize(256),
      transforms.RandomHorizontalFlip(),
      transforms.CenterCrop(224),
      transforms.ToTensor(),
      transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
  ])

  # Example of loading an image (replace with your own image path/loading method)
  # For demonstration, we use a sample image URL
  try:
      img = Image.open(img_path)
  except:
      print("Could not load image from URL, using a placeholder.")
      img = Image.new('RGB', (224, 224), color = 'red') # Placeholder if image fails to load

  # Apply transformations and prepare for model input
  img_t = preprocess(img)
  # Add batch dimension: PyTorch expects input as [batch_size, channels, height, width]
  input_tensor = img_t.unsqueeze(0)
  return input_tensor


## Load the CIFAR-10 DATASET

#### Using torchvision

In [4]:
import torch
import torchvision
import torchvision.transforms as transforms

# Define transformation
transform = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))])

# Download and load training dataset
trainset = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=transform)
trainloader = torch.utils.data.DataLoader(trainset, batch_size=4, shuffle=True, num_workers=2)

# Download and load test dataset
testset = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=transform)
testloader = torch.utils.data.DataLoader(testset, batch_size=4, shuffle=False, num_workers=2)

classes = ('plane', 'car', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck')

print('CIFAR-10 dataset downloaded and loaded successfully.')

100%|██████████| 170M/170M [01:26<00:00, 1.98MB/s]


CIFAR-10 dataset downloaded and loaded successfully.


## Get the RESNET18 Model

---



### Load Resnet18 with Pre-trained Weights on the ImageNet 1K dataset

In [5]:
# Load the ResNet-18 model with default ImageNet1K V1 weights
resnet18_model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 195MB/s]


#### Test Pretrained Resnet18

In [6]:
# Set the model to evaluation mode (optional, good practice for inference)
resnet18_model.eval()

ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
  

In [7]:
path = "./goldfish.jpg"
input_tensor = preprocess_image(path)

In [8]:
with torch.no_grad():
    output = resnet18_model(input_tensor)


In [9]:
_, index = torch.max(output, 1)
# You would typically load the ImageNet class labels (e.g., from a text file)
# to get the human-readable name of the class
print(f"Predicted class index: {index.item()}")


Predicted class index: 1
